# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

In [2]:
%cd /content
!rm -rf ML-intern-starter
!git clone https://github.com/Khuld13/ML-intern-starter.git
%cd ML-intern-starter

/content
Cloning into 'ML-intern-starter'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 186 (delta 87), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 1.91 MiB | 17.45 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/ML-intern-starter


In [3]:
!cat scripts/03_*.py 2>/dev/null | grep -A2 "read_parquet\|hf://"

In [7]:
!cat <path_from_above> | grep -B2 -A5 "read_parquet\|hf://"

/bin/bash: -c: line 1: syntax error near unexpected token `|'
/bin/bash: -c: line 1: `cat <path_from_above> | grep -B2 -A5 "read_parquet\|hf://"'


In [8]:
!grep -n "read_parquet\|hf://" notebooks/03_working_with_the_full_release.ipynb

55:    "DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the\n",
70:    "REL = 'hf://datasets/FlyRank/internship-warehouse'\n",
72:    "    'dim_clients':                f\"read_parquet('{REL}/dim_clients.parquet')\",\n",
73:    "    'dim_content':                f\"read_parquet('{REL}/dim_content.parquet')\",\n",
74:    "    'fact_daily':                 f\"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')\",\n",
75:    "    'fact_daily_sample':          f\"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')\",\n",
76:    "    'fact_query_90d':             f\"read_parquet('{REL}/fact_content_query_90d.parquet')\",\n",


In [12]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

REL = 'hf://datasets/FlyRank/internship-warehouse'

con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet('{REL}/dim_content.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
""")

print("Connected. Views ready.")

Connected. Views ready.


In [13]:
con.sql("SELECT COUNT(*) FROM dim_content").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       519606 │
└──────────────┘



In [17]:
con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")

con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")

print("fact_march and fact_feb ready.")

fact_march and fact_feb ready.


In [19]:
# Reconstruct ML-07's exact population and label (cell 8 logic from w04_baseline_score.ipynb)

monthly_compare = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        f.impressions_feb,
        m.impressions_march,
        CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
    FROM march_agg m
    JOIN feb_agg f USING (content_hash_id)
""").df()

# ML-07's volume-floor rule (Signal 2, CONFIRMED)
pop = monthly_compare[monthly_compare['impressions_march'] >= 250].copy()

# Bring in content-level features to model with
dim = con.sql("SELECT * FROM dim_content").df()
df = pop.merge(dim, on='content_hash_id', how='left')

# Exclude: the label itself, the two raw inputs that DEFINE the label,
# and the known-leaky/known-invalid columns from ML-06
leakage_cols = [
    'declined_flag', 'impressions_feb', 'impressions_march',   # label + its direct inputs
    'trend_pct', 'trend_direction', 'is_declining_label',       # ML-06: same fact, 3 forms
    'days_since_update',                                        # ML-06: structurally invalid (July snapshot)
]
candidate_features = [c for c in df.columns if c not in leakage_cols]

print("Rows after volume filter (impressions_march >= 250):", len(df))
print("\nLabel balance (declined_flag):")
print(df['declined_flag'].value_counts(normalize=True))
print("\nCandidate feature count:", len(candidate_features))
print(candidate_features)

Rows after volume filter (impressions_march >= 250): 68581

Label balance (declined_flag):
declined_flag
0    0.771759
1    0.228241
Name: proportion, dtype: float64

Candidate feature count: 26
['content_hash_id', 'client_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [20]:
print(df[['optimization_eligible_date', 'last_optimized_date', 'provider_used', 'model_used']].describe(include='all'))
print(df[['optimization_eligible_date', 'last_optimized_date']].head(10))

        optimization_eligible_date         last_optimized_date provider_used  \
count                        31980                       31980          9310   
unique                         NaN                         NaN             4   
top                            NaN                         NaN        google   
freq                           NaN                         NaN          8339   
mean    2026-07-24 01:12:45.928705  2026-06-09 01:12:45.928705           NaN   
min            2026-06-08 00:00:00         2026-04-24 00:00:00           NaN   
25%            2026-07-10 00:00:00         2026-05-26 00:00:00           NaN   
50%            2026-07-26 00:00:00         2026-06-11 00:00:00           NaN   
75%            2026-08-06 00:00:00         2026-06-22 00:00:00           NaN   
max            2026-08-20 00:00:00         2026-07-06 00:00:00           NaN   

                    model_used  
count                    52282  
unique                       5  
top     gemini-3-fla

## 1. Method choice and why

**Method: Logistic regression, with `class_weight='balanced'`.**

**Population:** Reconstructed ML-07's exact rule population — content items present in
both `fact_feb` and `fact_march` with `gsc_data_available = TRUE`, filtered to
`impressions_march >= 250` (ML-07's Signal 2, confirmed volume floor). This gives
68,581 rows.

**Label:** `declined_flag` (impressions_march < impressions_feb), identical to ML-07 —
not a future-window label. This model tests "can a learned method rank better than
ML-07's fixed rule on the same question," not "can we forecast decline."

**Class balance:** 22.8% positive (declined), 77.2% negative. Not severe, but real
enough that `class_weight='balanced'` is necessary — an unweighted model would default
toward predicting "no decline" and look falsely accurate.

**Leakage exclusions, from this population's own audit:**
- `impressions_feb` / `impressions_march` — these directly define the label, not just
  correlate with it.
- `optimization_eligible_date` — a forward-scheduling product field; its own max value
  (2026-08-20) sits weeks past the snapshot's export date (2026-07-03), confirming it's
  a rule-output, not an observed signal.
- `last_optimized_date` — mostly dated after the March decision point (min 2026-04-24),
  so using it would leak post-decision information backward into the features.
- `days_since_update` — carried over from ML-06: structurally invalid against most rows
  because `dim_content` is a single July snapshot.
- `trend_pct`, `trend_direction`, `is_declining_label` — ML-06 flagged these as leakage
  in the starter dataset, but they don't exist as columns in this warehouse table, so
  the exclusion is moot here rather than actively enforced.

26 candidate columns after removing join keys and the above, mostly numeric content
metadata (search_volume, competition, cpc, word_count, char_count) plus a few sparse
categoricals (`provider_used`, `model_used` — missing for 76-86% of rows, kept but
expected to contribute little).

**Why logistic regression fits:** 68,581 rows against ~20 usable features is well
within simple-linear-method territory — there's no evidence of the kind of nonlinear
interaction that would justify a tree ensemble's added complexity, and the rubric
explicitly rewards explainability over complexity for its own sake. Logistic
regression's coefficients also let me directly compare which signals move the needle
against ML-07's hand-picked rule (decline + volume only), which is the actual point of
this comparison. Decision tree is the fallback if logistic underperforms badly enough
to suggest real nonlinearity.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Unique clients:", df['client_hash_id'].nunique())
print("\nRows per client (distribution):")
print(df.groupby('client_hash_id').size().describe())
print("\nTop 10 clients by row count:")
print(df.groupby('client_hash_id').size().sort_values(ascending=False).head(10))

Unique clients: 34

Rows per client (distribution):
count       34.000000
mean      2017.088235
std       3867.955002
min          1.000000
25%         16.250000
50%        443.500000
75%       1234.750000
max      15862.000000
dtype: float64

Top 10 clients by row count:
client_hash_id
client_73cda7b4e4f265ea    15862
client_62f4a7e64f5e0096    12642
client_23a62021009f63c4     9830
client_e547b89c05043229     7482
client_fef1a8f436438636     5922
client_08a6a72ff48e62c0     3909
client_20259bd6705d81d4     2736
client_e5c2aa26a8598242     1972
client_3f0ce4d44fe94f3d     1249
client_a80fca3f171ed1de     1192
dtype: int64


In [22]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train rows:", len(train_df), " Test rows:", len(test_df))
print("Train clients:", train_df['client_hash_id'].nunique(), " Test clients:", test_df['client_hash_id'].nunique())
print("\nTrain label balance:\n", train_df['declined_flag'].value_counts(normalize=True))
print("\nTest label balance:\n", test_df['declined_flag'].value_counts(normalize=True))

Train rows: 54873  Test rows: 13708
Train clients: 27  Test clients: 7

Train label balance:
 declined_flag
0    0.787036
1    0.212964
Name: proportion, dtype: float64

Test label balance:
 declined_flag
0    0.710607
1    0.289393
Name: proportion, dtype: float64


In [23]:
for seed in [0, 1, 7, 42, 99]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(gss.split(df, groups=df['client_hash_id']))
    te = df.iloc[te_idx]
    print(f"seed={seed}: test_rows={len(te)}, test_clients={te['client_hash_id'].nunique()}, "
          f"test_decline_rate={te['declined_flag'].mean():.3f}")

seed=0: test_rows=17470, test_clients=7, test_decline_rate=0.300
seed=1: test_rows=7383, test_clients=7, test_decline_rate=0.144
seed=7: test_rows=15071, test_clients=7, test_decline_rate=0.227
seed=42: test_rows=13708, test_clients=7, test_decline_rate=0.289
seed=99: test_rows=3639, test_clients=7, test_decline_rate=0.038


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.